# Case study: imbalance strategies for rare headings

*Session 9, block 3 practice. Author: course team, licence CC-BY-4.0.*

Two tasks:

- **Part A, the long tail.** All headings of the 50,000-decision sample, a linear classifier on the TF-IDF matrix of the descriptions, time-based validation (2017–2021 → 2022–2023). Strategies: none, balanced class weights, oversampling rare headings to a minimum count, grouping rare headings.
- **Part B, the footwear task.** The six headings of chapter 64, 50 dense text components, five strategies inside imbalanced-learn pipelines, with a logistic regression and a random forest, and per-class precision and recall.

Theory: [page 3](../theory/03-class-imbalance.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"
from collections import Counter

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

# Part A: the long tail of headings

In [ ]:
decisions = pd.read_parquet(DATA / "train_sample.parquet")
fit = (decisions["start_date"] < "2022-01-01").to_numpy()
val = ~fit
y = decisions["heading"].to_numpy()
counts_fit = Counter(y[fit])
print("headings in the fitting rows:", len(counts_fit),
      "| with fewer than 5 rows:", sum(n < 5 for n in counts_fit.values()))
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_fit = tfidf.fit_transform(decisions.loc[fit, "description"])
X_val = tfidf.transform(decisions.loc[val, "description"])


def linear(class_weight=None):
    return SGDClassifier(loss="hinge", alpha=1e-5, class_weight=class_weight, random_state=0, n_jobs=-1)

In [ ]:
rows = []


def record(name, pred):
    rows.append({"strategy": name, "accuracy": accuracy_score(y[val], pred),
                 "macro-F1": f1_score(y[val], pred, average="macro", zero_division=0),
                 "headings predicted": len(set(pred))})


record("no correction", linear().fit(X_fit, y[fit]).predict(X_val))
record("balanced class weights", linear("balanced").fit(X_fit, y[fit]).predict(X_val))
for minimum in (20, 50):
    sampler = RandomOverSampler(sampling_strategy={c: max(n, minimum) for c, n in counts_fit.items()},
                                random_state=0)
    X_res, y_res = sampler.fit_resample(X_fit, y[fit])          # fitting rows only
    record(f"oversample to >= {minimum}", linear().fit(X_res, y_res).predict(X_val))
grouped = np.where([counts_fit[h] < 5 for h in y[fit]], "rare", y[fit])
record("group headings < 5 rows", linear().fit(X_fit, grouped).predict(X_val))
pd.DataFrame(rows).set_index("strategy").round(3)

**Questions.** Which strategy changes macro-F1 most, and in which direction? Why can a model that predicts the class "rare" never be right on the leaderboard?

# Part B: the footwear task (chapter 64)

In [ ]:
shoes = decisions[decisions["chapter"] == "64"].reset_index(drop=True)
y_shoes = shoes["heading"]
print(y_shoes.value_counts().to_dict())
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True)
X_shoes = pd.DataFrame(TruncatedSVD(50, random_state=0).fit_transform(char_tfidf.fit_transform(shoes["description"])),
                       columns=[f"svd{i}" for i in range(50)])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
labels = sorted(y_shoes.unique())

In [ ]:
def strategies(make_model, weighted_model):
    return {
        "no correction": make_pipeline(StandardScaler(), make_model()),
        "undersampling": make_pipeline(StandardScaler(), RandomUnderSampler(random_state=0), make_model()),
        "oversampling": make_pipeline(StandardScaler(), RandomOverSampler(random_state=0), make_model()),
        "SMOTE": make_pipeline(StandardScaler(), SMOTE(random_state=0), make_model()),
        "class weights": make_pipeline(StandardScaler(), weighted_model()),
    }


models = {
    "logistic regression": strategies(lambda: LogisticRegression(max_iter=2000),
                                      lambda: LogisticRegression(max_iter=2000, class_weight="balanced")),
    "random forest": strategies(lambda: RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
                                # "balanced_subsample": weights per bootstrap sample; scikit-learn 1.9.1
                                # fails with "balanced" on labels such as "6401" in random forests
                                lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample",
                                                               random_state=0, n_jobs=-1)),
}
rows, predictions = [], {}
for model_name, cands in models.items():
    for name, pipe in cands.items():
        pred = cross_val_predict(pipe, X_shoes, y_shoes, cv=cv)   # samplers run on training folds only
        predictions[(model_name, name)] = pred
        p, r, f, _ = precision_recall_fscore_support(y_shoes, pred, labels=labels, zero_division=0)
        rows.append({"model": model_name, "strategy": name, "accuracy": accuracy_score(y_shoes, pred),
                     "macro-F1": f.mean(), "6401 recall": r[0], "6406 precision": p[5], "6406 recall": r[5]})
pd.DataFrame(rows).set_index(["model", "strategy"]).round(3)

In [ ]:
print(classification_report(y_shoes, predictions[("random forest", "no correction")], digits=3, zero_division=0))

**Questions.**

1. Do the corrections help the logistic regression? The random forest?
2. Which strategy would you recommend to a customs office that wants to find as many "parts of footwear" (6406) decisions as possible for a manual check? Which for the best macro-F1?
3. Write one sentence per strategy in the form "recall of 6406 rose from … to …, precision fell from … to …".

## Exercises

1. In Part A, try `sampling_strategy` minimum counts of 5 and 100. Is there a best value?
2. In Part B, replace the stratified folds by a time-based split (fit 2017–2021, validate 2022–2023). How many 6401 decisions are left in the validation period, and what does that mean for the reliability of the per-class numbers?